# 03 — Training  ⏱️ SLOW — run this the night before the demo

This is the step that takes real time (many minutes to hours depending on
subset size / epochs / GPU). **Run it ahead of time** and let it save
checkpoints to `../checkpoints/`. During the live demo you don't re-run this
notebook — you just load the checkpoint it produced in `04_decomposition`.

Crash-recovery: training resumes automatically from `yolov3_last.pt` +
its JSON sidecar if this notebook (or the machine) is interrupted, so a
long run is safe to leave going overnight.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath("../src"))
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from model import YOLOv3
from dataset import CocoSubsetDataset, yolo_collate_fn
from train_utils import fit

with open("../checkpoints/run_config.json") as f:
    cfg = json.load(f)
CLASS_NAMES, IMG_SIZE, DATA_ROOT, BATCH_SIZE = cfg["class_names"], cfg["img_size"], cfg["data_root"], cfg["batch_size"]
NUM_CLASSES = len(CLASS_NAMES)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## Training config — the knobs you'd actually tune before the demo run

In [ ]:
EPOCHS = 120
LEARNING_RATE = 1e-3   # higher base LR; the warmup+cosine schedule handles it
CKPT_DIR = "../checkpoints"
IMAGES_PER_CLASS = 2000   # match notebook 01

train_ds = CocoSubsetDataset(DATA_ROOT, "train2017", CLASS_NAMES, img_size=IMG_SIZE,
                              images_per_class=IMAGES_PER_CLASS, augment=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=yolo_collate_fn, num_workers=0)

model = YOLOv3(num_classes=NUM_CLASSES)

# --- Pretrained backbone (ImageNet Darknet-53 via timm) ---
# Initializes model.backbone from pretrained weights; detection neck/heads
# stay random and train from scratch. This is the single biggest lever for
# reaching good mAP. Set USE_PRETRAINED_BACKBONE=False to train fully from
# scratch (much harder to converge to high mAP).
USE_PRETRAINED_BACKBONE = True
if USE_PRETRAINED_BACKBONE:
    from pretrained_backbone import load_pretrained_backbone
    load_pretrained_backbone(model)

model = model.to(DEVICE)

In [ ]:
history = fit(model, train_loader, DEVICE, epochs=EPOCHS, lr=LEARNING_RATE,
              ckpt_dir=CKPT_DIR, num_classes=NUM_CLASSES, resume=True,
              warmup_epochs=3, use_schedule=True)

## Loss curve — sanity check that training actually converged before the demo

In [ ]:
losses = [h["loss"] for h in history]
plt.figure(figsize=(8, 4))
plt.plot(losses, marker="o")
plt.xlabel("epoch"); plt.ylabel("avg training loss"); plt.title("YOLOv3 training loss")
plt.grid(alpha=0.3)
plt.show()
print(f"final loss: {losses[-1]:.4f}  (best: {min(losses):.4f})")

## Output

`../checkpoints/yolov3_best.pt` — the trained weights the decomposition
notebook loads. `../checkpoints/history.json` — full loss history if you
want to show the convergence plot live without re-running training.